# BrakeGuard - research prototype (RESEARCH CODE)

This notebook is the *research* version of the brake-fault model, written the way exploratory
notebooks usually are. It is kept **on purpose** to contrast with the production package
`brakeguard/` (see the report, section 1.2). Typical research-code traits are marked `# (R#)`:

* (R1) global mutable state, cells must be run in order
* (R2) magic numbers and hard-coded paths
* (R3) `print` instead of logging, no error handling
* (R4) no input validation / schema
* (R5) training and inference logic copy-pasted, no tests
* (R6) a scale-dependent explanation heuristic that nobody reviewed

In [1]:
import numpy as np, joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score, average_precision_score

In [2]:
# (R1)(R2) globals + magic numbers
n = 3000
rng = np.random.default_rng(42)
temp = rng.normal(155, 35, n).clip(70, 280)
pressure = rng.normal(65, 15, n).clip(20, 110)
wear = rng.beta(2.0, 2.5, n) * 100
vibration = rng.gamma(2.0, 0.8, n).clip(0.1, 8)
fluid_age = rng.gamma(2.2, 120, n).clip(5, 900)
odo = rng.uniform(5000, 180000, n)
score = 0.022*(temp-120) + 0.028*np.abs(pressure-65) + 0.065*(wear-35) + 0.55*(vibration-1.2) + 0.0022*(fluid_age-180) + 0.000008*(odo-60000)
y = (rng.random(n) < 1/(1+np.exp(-(score-2.2)))).astype(int)
X = np.column_stack([temp, pressure, wear, vibration, fluid_age, odo])
X.shape, y.mean()

((3000, 6), np.float64(0.515))

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=160, max_depth=8, min_samples_leaf=4, random_state=42, class_weight="balanced")
model.fit(X_train, y_train)
p = model.predict_proba(X_test)[:, 1]
pred = (p >= 0.5).astype(int)
# (R3) prints, metrics computed by hand each time
print("recall", recall_score(y_test, pred))
print("precision", precision_score(y_test, pred))
print("f1", f1_score(y_test, pred))
print("pr_auc", average_precision_score(y_test, p))

recall 0.7961165048543689
precision 0.7760252365930599
f1 0.7859424920127795
pr_auc 0.8521362125943502


In [4]:
# (R2) hard-coded relative path - breaks when run from another directory
joblib.dump(model, "model_research.joblib")

['model_research.joblib']

In [5]:
# (R4)(R5) inference: feature order implicit, no validation, thresholds copy-pasted
vehicle = [186.4, 72.1, 78.0, 3.2, 410, 84250]
prob = model.predict_proba([vehicle])[0, 1]
risk = "HIGH" if prob >= 0.70 else "MEDIUM" if prob >= 0.40 else "LOW"
print(prob, risk)

0.9612534171855547 HIGH


In [6]:
# (R6) explanation = importance * |raw value| -> odometer (tens of thousands of km) always "wins"
names = ["brake_temperature", "brake_pressure", "pad_wear_pct", "vibration_rms", "fluid_age_days", "odometer_km"]
sorted(zip(names, model.feature_importances_ * np.abs(vehicle)), key=lambda t: -t[1])[:3]

[('odometer_km', np.float64(7680.767793791109)),
 ('fluid_age_days', np.float64(38.70121692123383)),
 ('pad_wear_pct', np.float64(35.455368238298306))]

In [7]:
# (R4) nothing stops physically impossible input: pad wear of 450 % is happily scored
model.predict_proba([[186.4, 72.1, 450.0, 3.2, 410, 84250]])[0, 1]

np.float64(0.94051369724837)

In [8]:
import os
os.remove("model_research.joblib")  # tidy up the side effect of the cell above